# Analysis

This notebook will contain all the analysis of customer's info, product info, and sales info of a bicycle company.

In [1]:
import duckdb

conn = duckdb.connect('final_db.db')

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaylimit = None

displaylimit: Value None will be treated as 0 (no limit)

In [2]:
%%sql
-- Check database tables
    
SELECT table_name, column_name
FROM information_schema.columns

Running query in 'duckdb'

table_name,column_name
bought_prd,product_key
bought_prd,product_id
bought_prd,product_number
bought_prd,product_name
bought_prd,category_id
bought_prd,category
bought_prd,subcategory
bought_prd,maintenance
bought_prd,cost
bought_prd,product_line


## Customer Behaviour

In [3]:
%%sql
-- Derive ages of customers and obtain their range
    
SELECT 
    MIN((today() - birthdate)/365) AS min_age, 
    AVG((today() - birthdate)/365) AS average_age, 
    MAX((today() - birthdate)/365) AS max_age
FROM dim_customers;

Running query in 'duckdb'

min_age,average_age,max_age
39.78904109589041,56.95762696526546,110.20821917808219


In [4]:
%%sql
-- Create temporary view of age category
    
CREATE OR REPLACE VIEW age_table AS
SELECT 
    customer_id,
    CASE
        WHEN (today() - birthdate)/365 < 40 THEN '<40'
        WHEN (today() - birthdate)/365 BETWEEN 41 AND 50 THEN '41-50'
        WHEN (today() - birthdate)/365 BETWEEN 51 AND 60 THEN '51-60'
        WHEN (today() - birthdate)/365 BETWEEN 61 AND 70 THEN '61-70'
        WHEN (today() - birthdate)/365 BETWEEN 71 AND 80 THEN '71-80'
        WHEN (today() - birthdate)/365 BETWEEN 81 AND 90 THEN '81-90'
        WHEN (today() - birthdate)/365 BETWEEN 91 AND 100 THEN '91-100'
        ELSE '100+'
    END AS age_category,
FROM dim_customers;

Running query in 'duckdb'

Count


In [5]:
%%sql
-- Group customers by age category
    
SELECT age_category, COUNT(*) AS total
FROM age_table
GROUP BY age_category
ORDER BY COUNT(*) DESC;

Running query in 'duckdb'

age_category,total
41-50,5603
51-60,5093
61-70,3326
100+,1988
71-80,1813
81-90,534
<40,78
91-100,49


Majority of our customers are aged 41-70, with a surprising number of customers aged 100 or above. The minimum age is 39, while the maximum age is 110.

Now, we wanted to know total spending based on age category.

In [6]:
%%sql

SELECT 
    a.age_category, 
    MIN(s.sales_amount) AS min_amount, 
    AVG(s.sales_amount) AS average_amount, 
    MAX(s.sales_amount) AS max_amount, 
    SUM(s.sales_amount) AS total_amount
FROM age_table AS a
JOIN fact_sales AS s
USING (customer_id)
GROUP BY age_category
ORDER BY SUM(s.sales_amount) DESC;

Running query in 'duckdb'

age_category,min_amount,average_amount,max_amount,total_amount
51-60,2,526.5094262054384,3578,8964876
41-50,2,483.0051208633886,3578,8771856
61-70,2,501.52012721490235,3578,5519229
100+,2,489.51729970782714,3578,3183331
71-80,2,408.56286802909597,3578,2359042
81-90,2,272.95533980582525,3578,421716
<40,2,456.9502074688797,3578,110125
91-100,2,191.1851851851852,3375,25810


Customers aged 51 to 60 spend the most on average and made up the most in our customer's demographics.

In [7]:
# Drop the temporary view

%sql DROP VIEW age_table

Running query in 'duckdb'

Success


In [8]:
%%sql
-- Check customer's genders
    
SELECT gender, COUNT(gender)
FROM dim_customers
GROUP BY gender;

Running query in 'duckdb'

gender,count(gender)
Female,6848
Male,7067
N/A,4569


In [9]:
%%sql
-- Check marital status of customers
    
SELECT marital_status, COUNT(marital_status)
FROM dim_customers
GROUP BY marital_status;

Running query in 'duckdb'

marital_status,count(marital_status)
Married,10011
Single,8473


There appears little difference between numbers of male and female and their martal status.

In [10]:
%%sql
-- Check country of purchases
    
SELECT c.country, SUM(s.sales_amount) AS total_amount
FROM dim_customers AS c
JOIN fact_sales AS s
USING (customer_id)
GROUP BY country
ORDER BY SUM(s.sales_amount) DESC;

Running query in 'duckdb'

country,total_amount
United States,9162311
Australia,9060058
United Kingdom,3391351
Germany,2894066
France,2643741
Canada,1977638
N/A,226820


Customers from United States and Australia made up the most in our customer's demographics

## Product Performance

In [11]:
%%sql
-- Check for unsold products
    
SELECT COUNT(*)
FROM dim_products
LEFT JOIN fact_sales
USING (product_key)
WHERE fact_sales.order_number IS NULL
LIMIT 10;

Running query in 'duckdb'

count_star()
95


There are a total of 95 products currently in production and not bought by any of our customers.

In [12]:
%%sql
-- Check for bought products not recorded in product table
    
SELECT COUNT(*)
FROM fact_sales
LEFT JOIN dim_products
USING (product_key)
WHERE dim_products.product_id IS NULL;

Running query in 'duckdb'

count_star()
3238


There are 3238 bought products not recorded in our database. 

For analysis purpose, we will ignore any unrecorded products.

We now create a temporary view of products not being bought.

In [13]:
%%sql

CREATE OR REPLACE VIEW not_bought_prd AS
SELECT dim_products.*
FROM dim_products
LEFT JOIN fact_sales
USING (product_key)
WHERE fact_sales.order_number IS NULL

Running query in 'duckdb'

Count


In [14]:
%%sql
-- Check total number of bought products by subcategory
    
SELECT b.subcategory, SUM(s.quantity) AS total_bought
FROM dim_products AS b
JOIN fact_sales AS s
USING (product_key)
GROUP BY subcategory
ORDER BY SUM(s.quantity) DESC;

Running query in 'duckdb'

subcategory,total_bought
Tires and Tubes,17336
Bottles and Cages,7995
Helmets,6441
Road Bikes,5226
Mountain Bikes,4574
Jerseys,3334
Caps,2190
Touring Bikes,2167
Fenders,2121
Gloves,1430


Tires and tubes were bought the most, which is to be expected for frequent bicycle users.

Now check for products that were not bought at all.

In [15]:
%%sql

SELECT category, subcategory, COUNT(category)
FROM not_bought_prd
GROUP BY category, subcategory
ORDER BY category, subcategory;

Running query in 'duckdb'

category,subcategory,count(category)
Components,Bottom Brackets,3
Components,Brakes,2
Components,Chains,1
Components,Cranksets,3
Components,Derailleurs,2
Components,Handlebars,8
Components,Mountain Frames,20
Components,Road Frames,22
Components,Saddles,9
Components,Touring Frames,18


All products not bought are bicycle components. Perhaps components require mechanics knowledge not common among bicycle users. 

In [16]:
%%sql
-- Check costs of products
    
SELECT category, subcategory, MIN(cost) AS min_cost, CAST(AVG(cost) AS DECIMAL(18, 2)) AS average_cost, MAX(cost) AS max_cost
FROM dim_products
GROUP BY category, subcategory
ORDER BY category, subcategory;

Running query in 'duckdb'

category,subcategory,min_cost,average_cost,max_cost
Accessories,Bike Racks,45,45.00,45
Accessories,Bike Stands,59,59.00,59
Accessories,Bottles and Cages,2,3.00,4
Accessories,Cleaners,3,3.00,3
Accessories,Fenders,8,8.00,8
Accessories,Helmets,13,13.00,13
Accessories,Hydration Packs,21,21.00,21
Accessories,Tires and Tubes,1,7.18,13
Bikes,Mountain Bikes,295,612.45,1266
Bikes,Road Bikes,344,947.11,1555


As expected, bikes cost the most on average. We should also stop the production of bicycle components are they are costly, or at least sell to mechanics.

In [17]:
# Drop any temporary views

%sql DROP VIEW not_bought_prd

Running query in 'duckdb'

Success


## Sales Trend

First, we analyze durations between order, shipping, and due dates by creating a temporary view.

In [18]:
%%sql

CREATE OR REPLACE VIEW duration AS

SELECT 
    order_number,
    shipping_date - order_date AS order_to_shipping,
    due_date - shipping_date AS shipping_to_due,
    quantity
FROM fact_sales;

Running query in 'duckdb'

Count


In [19]:
%%sql

SELECT quantity, MIN(order_to_shipping), AVG(order_to_shipping), MAX(order_to_shipping)
FROM duration
GROUP BY quantity
ORDER BY quantity;

Running query in 'duckdb'

quantity,min(order_to_shipping),avg(order_to_shipping),max(order_to_shipping)
1,"7 days, 0:00:00","7 days, 0:00:00","7 days, 0:00:00"
2,"7 days, 0:00:00","7 days, 0:00:00","7 days, 0:00:00"
3,"7 days, 0:00:00","7 days, 0:00:00","7 days, 0:00:00"
4,"7 days, 0:00:00","7 days, 0:00:00","7 days, 0:00:00"
5,"7 days, 0:00:00","7 days, 0:00:00","7 days, 0:00:00"
10,"7 days, 0:00:00","7 days, 0:00:00","7 days, 0:00:00"


In [20]:
%%sql

SELECT quantity, MIN(shipping_to_due), AVG(shipping_to_due), MAX(shipping_to_due)
FROM duration
GROUP BY quantity
ORDER BY quantity;

Running query in 'duckdb'

quantity,min(shipping_to_due),avg(shipping_to_due),max(shipping_to_due)
1,"5 days, 0:00:00","5 days, 0:00:00","5 days, 0:00:00"
2,"5 days, 0:00:00","5 days, 0:00:00","5 days, 0:00:00"
3,"5 days, 0:00:00","5 days, 0:00:00","5 days, 0:00:00"
4,"5 days, 0:00:00","5 days, 0:00:00","5 days, 0:00:00"
5,"5 days, 0:00:00","5 days, 0:00:00","5 days, 0:00:00"
10,"5 days, 0:00:00","5 days, 0:00:00","5 days, 0:00:00"


It appears our delivery systems are working perfectly.

In [21]:
%sql DROP VIEW duration

Running query in 'duckdb'

Success


Finally, we analyze total profits of our business.

In [22]:
%%sql
-- Create profit tables by taking difference between product price and cost
    
CREATE OR REPLACE VIEW profit_table AS
SELECT 
    s.order_number, 
    s.product_key,
    s.order_date,
    p.category,
    p.subcategory,
    s.quantity,
    s.price,
    p.cost,
    s.price - p.cost AS profit
FROM fact_sales AS s
JOIN dim_products AS p
USING (product_key);

Running query in 'duckdb'

Count


In [23]:
%%sql

SELECT category, subcategory, SUM(profit) AS Total_Sales
FROM profit_table
GROUP BY category, subcategory
ORDER BY SUM(profit);

Running query in 'duckdb'

category,subcategory,Total_Sales
Clothing,Socks,3408
Clothing,Caps,4380
Accessories,Cleaners,4535
Clothing,Gloves,21450
Clothing,Vests,22480
Accessories,Bike Racks,24600
Accessories,Bike Stands,24900
Accessories,Hydration Packs,24922
Accessories,Fenders,29694
Accessories,Bottles and Cages,35136


Majority of profits are made by bicycles, while socks, caps, and cleaners made the least profits.

Now we analyze total profits over time.

In [24]:
%%sql

SELECT 
    EXTRACT('year' FROM order_date) AS year, 
    EXTRACT('month' FROM order_date) AS month, 
    SUM(profit*quantity) AS total_profit
FROM profit_table
GROUP BY EXTRACT('year' FROM order_date), EXTRACT('month' FROM order_date)
ORDER BY EXTRACT('year' FROM order_date), EXTRACT('month' FROM order_date);

Running query in 'duckdb'

year,month,total_profit
2011,12,9413
2012,1,104523
2012,2,113976
2012,3,84101
2012,4,97318
2012,5,78466
2012,6,150260
2012,7,113338
2012,8,134377
2012,9,125089


Our porfit flucturates on 2012 then steadily increases from 2013 onwards. This show our business is doing well.

In [25]:
%sql DROP VIEW profit_table

Running query in 'duckdb'

Success


In [26]:
conn.close()